# SmartAssist Feature Adoption Analysis

**Objective:** Analyze the adoption, engagement, retention impact, and support cost of the SmartAssist AI feature to inform the upcoming board meeting with actionable, data-driven recommendations.

**Datasets Used:**

| Dataset | Description |
|---------|-------------|
| `users.csv` | User profiles with account mapping |
| `subscriptions.csv` | Account-level subscription details (plan tier, MRR, status) |
| `feature_events.csv` | Feature usage event logs |
| `support_tickets.csv` | Customer support ticket records |

---

### Table of Contents
1. [Setup and Data Loading](#1-setup-and-data-loading)
2. [Data Exploration and Cleaning](#2-data-exploration-and-cleaning)
3. [Overall Adoption Rate](#3-overall-adoption-rate)
4. [Adoption by Plan Tier](#4-adoption-by-plan-tier)
5. [Repeat Usage Analysis](#5-repeat-usage-analysis)
6. [Churn Correlation](#6-churn-correlation)
7. [Support Cost Analysis](#7-support-cost-analysis)
8. [Executive Dashboard](#8-executive-dashboard)
9. [Executive Summary and Recommendations](#9-executive-summary-and-recommendations)

---
## 1. Setup and Data Loading

In [67]:
import os
os.chdir(r'C:\Users\Nikita Dhondji\Desktop\SmartAssist-AI-Impact-Analysis\notebooks')
print(os.getcwd())

C:\Users\Nikita Dhondji\Desktop\SmartAssist-AI-Impact-Analysis\notebooks


In [68]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# --- Configuration ---
COLORS = ["#1877F2", "#F0701A", "#5A24C7", "#E42C97", "#00487C", "#0EAC96"]
POSITIVE = "#63BE09"
NEGATIVE = "#D93616"

def styled_layout(title="", xaxis_title="", yaxis_title=""):
    """Return a consistent Plotly layout."""
    return dict(
        title=dict(text=title, font=dict(size=16, color='#1C1E21'), x=0.5, xanchor='center'),
        xaxis=dict(title=xaxis_title, gridcolor='#E4E6EB', showgrid=True),
        yaxis=dict(title=yaxis_title, gridcolor='#E4E6EB', showgrid=True),
        plot_bgcolor='white',
        font=dict(family='Helvetica, Arial, sans-serif', size=12),
        margin=dict(l=60, r=40, t=60, b=60),
    )

print("Libraries loaded and styling configured.")

Libraries loaded and styling configured.


In [69]:
# Load all 4 datasets
feature_events = pd.read_csv('../data/feature_events.csv')
subscriptions  = pd.read_csv('../data/subscriptions.csv')
support_tickets = pd.read_csv('../data/support_tickets.csv')
users          = pd.read_csv('../data/users.csv')

# Quick shape overview
shape_df = pd.DataFrame({
    'Dataset': ['feature_events', 'subscriptions', 'support_tickets', 'users'],
    'Rows': [feature_events.shape[0], subscriptions.shape[0], support_tickets.shape[0], users.shape[0]],
    'Columns': [feature_events.shape[1], subscriptions.shape[1], support_tickets.shape[1], users.shape[1]],
})
shape_df.style.hide(axis='index')

Dataset,Rows,Columns
feature_events,6467322,9
subscriptions,2500,16
support_tickets,5859,15
users,42777,13


---
## 2. Data Exploration and Cleaning

In [70]:
# Preview each dataset
for name, df in [('Users', users), ('Subscriptions', subscriptions),
                 ('Feature Events', feature_events), ('Support Tickets', support_tickets)]:
    print(f"\n{'=' * 60}")
    print(f"  {name}  --  {df.shape[0]:,} rows x {df.shape[1]} cols")
    print(f"{'=' * 60}")
    display(df.head(3))


  Users  --  42,777 rows x 13 cols


,user_id,account_id,email,full_name,role,job_title,department,license_type,signup_date,last_login_date,login_count,feature_adoption_score,is_active
0,1,1,gomezlisa@example.net,Christopher Tate,member,Marketing Manager,Finance,Full,2024-10-05,2024-12-29,123,61,True
1,2,1,martinezmitchell@example.org,Daniel Smith,member,Designer,Product,Full,2024-09-28,2024-12-31,18,80,True
2,3,1,erin53@example.org,Lori Stokes,member,Marketing Manager,Engineering,Limited,2024-10-06,2024-12-20,150,51,True



  Subscriptions  --  2,500 rows x 16 cols


,account_id,company_name,plan_tier,mrr,status,start_date,churn_date,upgrade_date,company_size,industry,acquisition_channel,sales_rep,contract_length_months,discount_percentage,region,payment_method
0,1,"Rodriguez, Figueroa and Sanchez",Pro,100.25,active,2024-09-07,NaN,NaN,11-50,Financial Services,Direct Sales,Alex Johnson,1,0.0,Europe,ACH
1,2,Doyle Ltd,Starter,29.60,active,2024-08-15,NaN,NaN,1-10,Media & Entertainment,Organic Search,None (Self-Serve),1,0.0,Europe,Credit Card
2,3,"Mcclain, Miller and Henderson",Starter,34.56,active,2024-08-22,NaN,NaN,1-10,Professional Services,Organic Search,None (Self-Serve),12,12.6,Latin America,Credit Card



  Feature Events  --  6,467,322 rows x 9 cols


,event_id,user_id,event_name,feature_name,event_date,session_id,device_type,duration_seconds,success
0,1,1,project_created,Core Product,2024-10-19 20:11:18,ce465118,Desktop,44,True
1,2,1,project_created,Core Product,2024-10-26 06:40:19,8b89e63c,Desktop,8,True
2,3,1,comment_added,Core Product,2024-11-01 16:41:15,6becbb25,Desktop,15,True



  Support Tickets  --  5,859 rows x 15 cols


,ticket_id,account_id,created_date,resolved_date,category,priority,channel,assigned_to,ticket_description,resolution_time_hours,first_response_time_hours,sentiment_score,csat_score,escalated,month
0,1,1,2024-11-19 13:30:28,2024-11-19 16:09:00.046458,Billing Question,Critical,In-App Chat,Agent_C,Protect serious under imagine adult degree who...,2.64,1.17,-0.04,2,False,2024-11
1,2,2,2024-10-15 17:59:05,2024-10-16 00:09:10.060885,SmartAssist,Medium,Email,Agent_B,Series seek world audience page local medical....,6.17,4.78,-0.19,1,True,2024-10
2,3,2,2024-10-28 11:32:37,2024-10-28 18:46:02.711880,SmartAssist,Critical,Email,Agent_B,Mother go while administration have. automatio...,7.22,1.63,-0.29,3,False,2024-10


In [71]:
# Check data types across all datasets
for name, df in [('Users', users), ('Subscriptions', subscriptions),
                 ('Feature Events', feature_events), ('Support Tickets', support_tickets)]:
    print(f"\n--- {name} dtypes ---")
    print(df.dtypes.to_string())


--- Users dtypes ---
user_id                   int64
account_id                int64
email                       str
full_name                   str
role                        str
job_title                   str
department                  str
license_type                str
signup_date                 str
last_login_date             str
login_count               int64
feature_adoption_score    int64
is_active                  bool

--- Subscriptions dtypes ---
account_id                  int64
company_name                  str
plan_tier                     str
mrr                       float64
status                        str
start_date                    str
churn_date                    str
upgrade_date              float64
company_size                  str
industry                      str
acquisition_channel           str
sales_rep                     str
contract_length_months      int64
discount_percentage       float64
region                        str
payment_method        

In [72]:
# Check for missing values
print("Missing Values Summary")
print("=" * 50)
for name, df in [('Subscriptions', subscriptions), ('Users', users),
                 ('Support Tickets', support_tickets), ('Feature Events', feature_events)]:
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print(f"\n{name}:")
        for col, count in missing.items():
            print(f"  - {col}: {count:,} missing ({count/len(df)*100:.1f}%)")
    else:
        print(f"\n{name}: No missing values")

Missing Values Summary

Subscriptions:
  - churn_date: 2,283 missing (91.3%)
  - upgrade_date: 2,500 missing (100.0%)

Users: No missing values

Support Tickets: No missing values

Feature Events: No missing values


In [73]:
# Inspect raw feature_name values -- identify naming inconsistencies
print("Raw feature_name distribution (top 10):")
print(feature_events['feature_name'].value_counts().head(10).to_string())

Raw feature_name distribution (top 10):
feature_name
Core Product    6433767
SmartAssist       23519
Smart Assist       5004
smart_assist       3390
SMARTASSIST        1642


In [74]:
# Normalize SmartAssist naming variants into a single clean label
def normalize_feature(name):
    """Collapse all SmartAssist spelling variants into one canonical label."""
    if pd.isna(name):
        return name
    cleaned = name.strip().lower().replace(' ', '').replace('_', '')
    return 'SmartAssist' if cleaned == 'smartassist' else name

feature_events['feature_name_clean'] = feature_events['feature_name'].apply(normalize_feature)

# Verify the fix
print("After normalization:")
print(feature_events['feature_name_clean'].value_counts().to_string())

After normalization:
feature_name_clean
Core Product    6433767
SmartAssist       33555


In [75]:
# Filter to SmartAssist events only
sa_events = feature_events[feature_events['feature_name_clean'] == 'SmartAssist'].copy()
print(f"Total SmartAssist events: {len(sa_events):,}")

Total SmartAssist events: 33,555


### Data Cleaning Complete

| Issue | Resolution |
|-------|-----------|
| `feature_name` had 4 spelling variants of "SmartAssist" | Normalized all to `SmartAssist` |
| `churn_date` has nulls | Expected -- represents active (non-churned) accounts |

---
## 3. Overall Adoption Rate

In [76]:
# Calculate overall SmartAssist adoption
total_users = users['user_id'].nunique()
ai_users = sa_events['user_id'].nunique()
non_ai_users = total_users - ai_users
adoption_rate = ai_users / total_users * 100

print(f"Total users:           {total_users:,}")
print(f"SmartAssist users:     {ai_users:,}")
print(f"Non-SmartAssist users: {non_ai_users:,}")
print(f"Adoption rate:         {adoption_rate:.1f}%")

Total users:           42,777
SmartAssist users:     5,472
Non-SmartAssist users: 37,305
Adoption rate:         12.8%


In [77]:
# Visualize adoption split
fig = go.Figure(data=[go.Bar(
    x=['Used SmartAssist', 'Did Not Use'],
    y=[ai_users, non_ai_users],
    marker_color=[COLORS[0], COLORS[5]],
    text=[f'{ai_users:,}', f'{non_ai_users:,}'],
    textposition='outside',
)])
fig.update_layout(
    **styled_layout(
        title=f'SmartAssist Adoption Rate -- {adoption_rate:.1f}%',
        yaxis_title='Number of Users',
    ),
    showlegend=False,
    height=450,
)
fig.show()

### Key Takeaway
- **Overall adoption is only ~12.8%** -- only ~5,500 out of ~42,800 users have tried SmartAssist.
- This is lower than expected and warrants deeper investigation by segment.

---
## 4. Adoption by Plan Tier

In [78]:
# Merge users with subscription tier info
users_with_tier = users.merge(
    subscriptions[['account_id', 'plan_tier', 'mrr']],
    on='account_id',
    how='left'
)

# Flag users who used SmartAssist
users_with_tier['used_smartassist'] = users_with_tier['user_id'].isin(sa_events['user_id'].unique())

# Calculate adoption rate per tier
tier_summary = (
    users_with_tier
    .groupby('plan_tier')
    .agg(total_users=('user_id', 'count'), ai_users=('used_smartassist', 'sum'))
    .reset_index()
)
tier_summary['adoption_rate'] = tier_summary['ai_users'] / tier_summary['total_users'] * 100

display(tier_summary.style.format({'adoption_rate': '{:.1f}%'}).hide(axis='index'))

plan_tier,total_users,ai_users,adoption_rate
Enterprise,16102,765,4.8%
Pro,15619,2490,15.9%
Starter,11056,2217,20.1%


In [79]:
# Visualize adoption by plan tier
fig = go.Figure(data=[go.Bar(
    x=tier_summary['plan_tier'],
    y=tier_summary['adoption_rate'],
    marker_color=COLORS[:len(tier_summary)],
    text=tier_summary['adoption_rate'].apply(lambda x: f'{x:.1f}%'),
    textposition='outside',
)])
fig.update_layout(
    **styled_layout(
        title='SmartAssist Adoption Rate by Plan Tier',
        xaxis_title='Plan Tier',
        yaxis_title='Adoption Rate (%)',
    ),
    showlegend=False,
    height=450,
)
fig.show()

### Key Takeaway
- **Starter** tier has the highest adoption (~20%).
- **Pro** tier follows at ~15.9%.
- **Enterprise** tier has the lowest adoption at only ~4.7%.

**Concerning:** Enterprise customers are our highest-value segment, yet they adopt SmartAssist the least. This may indicate the feature does not meet enterprise-grade needs (security, compliance, workflow fit).

---
## 5. Repeat Usage Analysis

In [80]:
# Count SmartAssist events per user
user_sa_counts = sa_events.groupby('user_id').size().reset_index(name='sa_event_count')

print(f"Total SmartAssist users: {len(user_sa_counts):,}")
print(f"\nUsage distribution:")
print(user_sa_counts['sa_event_count'].describe().to_string())

Total SmartAssist users: 5,472

Usage distribution:
count    5472.000000
mean        6.132127
std        12.543570
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max        72.000000


In [81]:
# Classify users into engagement segments
def classify_user(count):
    if count == 1:
        return 'One-Time (1 event)'
    elif count <= 9:
        return 'Light (2-9 events)'
    else:
        return 'Heavy (10+ events)'

user_sa_counts['segment'] = user_sa_counts['sa_event_count'].apply(classify_user)

segment_order = ['One-Time (1 event)', 'Light (2-9 events)', 'Heavy (10+ events)']
segment_counts = user_sa_counts['segment'].value_counts().reindex(segment_order)
segment_pcts = (segment_counts / segment_counts.sum() * 100).round(1)

# Display summary table
seg_df = pd.DataFrame({'Users': segment_counts, 'Percent': segment_pcts})
seg_df['Percent'] = seg_df['Percent'].apply(lambda x: f'{x:.1f}%')
display(seg_df)

,Users,Percent
segment,,
One-Time (1 event),4163,76.1%
Light (2-9 events),297,5.4%
Heavy (10+ events),1012,18.5%


In [82]:
# Visualize repeat usage distribution
fig = go.Figure(data=[go.Bar(
    x=segment_order,
    y=segment_counts.values,
    marker_color=COLORS[:3],
    text=[f'{v:,} ({p}%)' for v, p in zip(segment_counts.values, segment_pcts.values)],
    textposition='outside',
)])
fig.update_layout(
    **styled_layout(
        title='SmartAssist Repeat Usage Distribution',
        xaxis_title='User Segment',
        yaxis_title='Number of Users',
    ),
    showlegend=False,
    height=450,
)
fig.show()

### Key Takeaway
- **~76% of users** tried SmartAssist only once and never returned.
- Only ~18% qualify as heavy users (10+ events).
- Median usage is **1 event per user**.

**Major stickiness problem.** Most users try SmartAssist once but do not find enough value to return. The first-run experience needs significant improvement.

---
## 6. Churn Correlation

In [83]:
# Aggregate AI usage to account level
sa_with_account = sa_events.merge(users[['user_id', 'account_id']], on='user_id', how='left')
account_ai_usage = sa_with_account.groupby('account_id').size().reset_index(name='ai_events')

print(f"Accounts that used SmartAssist: {len(account_ai_usage):,}")

Accounts that used SmartAssist: 1,073


In [84]:
# Classify accounts by AI usage intensity
def classify_account(events):
    if events == 0:
        return 'No AI Usage'
    elif events < 10:
        return 'Light AI Usage'
    else:
        return 'Heavy AI Usage'

all_accounts = subscriptions[['account_id', 'status', 'mrr']].copy()
all_accounts = all_accounts.merge(account_ai_usage, on='account_id', how='left')
all_accounts['ai_events'] = all_accounts['ai_events'].fillna(0)
all_accounts['ai_segment'] = all_accounts['ai_events'].apply(classify_account)

# Calculate churn rate by segment
all_accounts['is_churned'] = (all_accounts['status'] == 'churned').astype(int)
overall_churn = all_accounts['is_churned'].mean() * 100

churn_by_segment = (
    all_accounts
    .groupby('ai_segment')
    .agg(total_accounts=('account_id', 'count'), churned_accounts=('is_churned', 'sum'))
    .reset_index()
)
churn_by_segment['churn_rate'] = churn_by_segment['churned_accounts'] / churn_by_segment['total_accounts'] * 100

print(f"Overall churn rate: {overall_churn:.1f}%\n")
display(churn_by_segment.style.format({'churn_rate': '{:.1f}%'}).hide(axis='index'))

Overall churn rate: 8.7%



ai_segment,total_accounts,churned_accounts,churn_rate
Heavy AI Usage,664,109,16.4%
Light AI Usage,409,44,10.8%
No AI Usage,1427,64,4.5%


In [85]:
# Visualize churn by AI usage segment
seg_order = ['No AI Usage', 'Light AI Usage', 'Heavy AI Usage']
churn_df = churn_by_segment.set_index('ai_segment').reindex(seg_order)

bar_colors = [POSITIVE, COLORS[1], NEGATIVE]

fig = go.Figure(data=[go.Bar(
    x=seg_order,
    y=churn_df['churn_rate'].values,
    marker_color=bar_colors,
    text=churn_df['churn_rate'].apply(lambda x: f'{x:.1f}%').values,
    textposition='outside',
)])
fig.add_hline(y=overall_churn, line_dash="dash", line_color=COLORS[0],
              annotation_text=f"Overall: {overall_churn:.1f}%", annotation_position="top right")
fig.update_layout(
    **styled_layout(
        title='Churn Rate by SmartAssist Usage Segment',
        xaxis_title='AI Usage Segment',
        yaxis_title='Churn Rate (%)',
    ),
    showlegend=False,
    height=450,
)
fig.show()

### Key Takeaway
- **No AI Usage:** ~4.5% churn (below baseline)
- **Light AI Usage:** ~10.8% churn (above baseline)
- **Heavy AI Usage:** ~16.4% churn (nearly 2x the overall rate)

**Counter-intuitive finding:** Heavier SmartAssist usage correlates with *higher* churn, not lower. This suggests the feature may be frustrating power users or failing to deliver on its promise.

---
## 7. Support Cost Analysis

In [86]:
# Identify AI-related support tickets
cat_filter = support_tickets['category'].str.lower().str.strip() == 'smartassist'
keyword_filter = support_tickets['ticket_description'].str.lower().str.contains(
    'smartassist|smart assist|ai feature|ai copilot', na=False
)
support_tickets['is_ai_related'] = cat_filter | keyword_filter

ai_tickets = support_tickets[support_tickets['is_ai_related']]
non_ai_tickets = support_tickets[~support_tickets['is_ai_related']]

print(f"Total tickets:   {len(support_tickets):,}")
print(f"AI-related:      {len(ai_tickets):,} ({len(ai_tickets)/len(support_tickets)*100:.1f}%)")
print(f"Non-AI-related:  {len(non_ai_tickets):,} ({len(non_ai_tickets)/len(support_tickets)*100:.1f}%)")

Total tickets:   5,859
AI-related:      1,738 (29.7%)
Non-AI-related:  4,121 (70.3%)


In [87]:
# Compare resolution time and CSAT between AI and non-AI tickets
avg_res_ai     = ai_tickets['resolution_time_hours'].mean()
avg_res_non_ai = non_ai_tickets['resolution_time_hours'].mean()
avg_csat_ai    = ai_tickets['csat_score'].mean()
avg_csat_non_ai = non_ai_tickets['csat_score'].mean()

comparison = pd.DataFrame({
    'Metric': ['Avg Resolution Time (hours)', 'Avg CSAT Score'],
    'AI Tickets': [f'{avg_res_ai:.1f}', f'{avg_csat_ai:.2f}'],
    'Non-AI Tickets': [f'{avg_res_non_ai:.1f}', f'{avg_csat_non_ai:.2f}'],
    'Difference': [f'{avg_res_ai - avg_res_non_ai:+.1f}h', f'{avg_csat_ai - avg_csat_non_ai:+.2f}'],
})
display(comparison.style.hide(axis='index'))

Metric,AI Tickets,Non-AI Tickets,Difference
Avg Resolution Time (hours),8.0,3.0,+4.9h
Avg CSAT Score,2.55,3.70,-1.16


In [88]:
# Visualize resolution time comparison
fig = go.Figure(data=[go.Bar(
    x=['AI Tickets', 'Non-AI Tickets'],
    y=[avg_res_ai, avg_res_non_ai],
    marker_color=[NEGATIVE, POSITIVE],
    text=[f'{avg_res_ai:.1f}h', f'{avg_res_non_ai:.1f}h'],
    textposition='outside',
)])
fig.update_layout(
    **styled_layout(
        title='Avg Resolution Time: AI vs Non-AI Tickets',
        yaxis_title='Hours',
    ),
    showlegend=False,
    height=450,
)
fig.show()

### Key Takeaway
- AI-related tickets take significantly longer to resolve (~2.7x longer).
- CSAT scores are lower for AI-related tickets.

**SmartAssist is creating a measurable support burden.** Customers are less satisfied and tickets take much longer to resolve, increasing support costs.

---
## 8. Executive Dashboard

In [89]:
# ── Color palette ──────────────────────────────────────────────
PURPLE       = '#534AB7'
PURPLE_MID   = '#7F77DD'
PURPLE_LIGHT = '#AFA9EC'
TEAL         = '#1D9E75'
CORAL        = '#D85A30'
RED          = '#E24B4A'
GRAY         = '#888780'
WHITE        = '#FFFFFF'
BG           = '#F8F8F8'
GRID         = '#E8E8E8'
TEXT_DARK    = '#1C1E21'
TEXT_MID     = '#5F5E5A'

# ── Tier colors (match tier_summary order) ─────────────────────
tier_colors_dash = [PURPLE, PURPLE_MID, PURPLE_LIGHT]

# ── Donut data from segment_pcts computed above ────────────────
donut_labels = ['One-Time', 'Light Repeat', 'Heavy']
donut_values = [
    float(segment_pcts.get('One-Time (1 event)', 76)),
    float(segment_pcts.get('Light (2-9 events)', 6)),
    float(segment_pcts.get('Heavy (10+ events)', 18)),
]
donut_colors = [PURPLE, TEAL, CORAL]

# ── Churn data ─────────────────────────────────────────────────
churn_colors_dash = [TEAL, GRAY, RED]

# ── KPI values ─────────────────────────────────────────────────
kpis = [
    (f'{adoption_rate:.1f}%', 'Overall Adoption',  f'{ai_users:,} of {total_users:,} users', TEXT_DARK),
    (f'{donut_values[2]:.0f}%',  'Heavy Users',        '10+ AI events',                          TEXT_DARK),
    (f'{churn_df["churn_rate"]["Heavy AI Usage"]:.1f}%', 'AI User Churn', f'vs {overall_churn:.1f}% overall', RED),
    (f'{donut_values[0]:.0f}%',  'One-Time Users',     'Low repeat engagement',                   TEXT_DARK),
]
kpi_x = [0.08, 0.31, 0.62, 0.85]

# ── Build 2x2 subplots ─────────────────────────────────────────
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Adoption Rate by Plan Tier',
        'Repeat Usage Distribution',
        'Churn Rate by AI Usage Segment',
        'Avg Resolution Time (hours)',
    ),
    vertical_spacing=0.10,
    horizontal_spacing=0.12,
    specs=[
        [{'type': 'xy'},     {'type': 'domain'}],
        [{'type': 'xy'},     {'type': 'xy'}],
    ],
)

# Top-left: Adoption by tier
fig.add_trace(go.Bar(
    x=tier_summary['plan_tier'],
    y=tier_summary['adoption_rate'],
    marker_color=tier_colors_dash,
    marker_line_width=0,
    text=tier_summary['adoption_rate'].apply(lambda x: f'{x:.1f}%'),
    textposition='outside',
    textfont=dict(size=12, color=TEXT_DARK),
    showlegend=False,
), row=1, col=1)

# Top-right: Repeat usage donut
fig.add_trace(go.Pie(
    labels=donut_labels,
    values=donut_values,
    hole=0.6,
    marker_colors=donut_colors,
    textinfo='label+percent',
    textfont=dict(size=11, color=TEXT_DARK),
    showlegend=False,
), row=1, col=2)

# Bottom-left: Churn by segment
fig.add_trace(go.Bar(
    x=seg_order,
    y=churn_df['churn_rate'].values,
    marker_color=churn_colors_dash,
    marker_line_width=0,
    text=churn_df['churn_rate'].apply(lambda x: f'{x:.1f}%').values,
    textposition='outside',
    textfont=dict(size=12, color=TEXT_DARK),
    showlegend=False,
), row=2, col=1)

# Bottom-right: Resolution time
fig.add_trace(go.Bar(
    x=['AI Tickets', 'Non-AI Tickets'],
    y=[avg_res_ai, avg_res_non_ai],
    marker_color=[RED, TEAL],
    marker_line_width=0,
    text=[f'{avg_res_ai:.1f}h', f'{avg_res_non_ai:.1f}h'],
    textposition='outside',
    textfont=dict(size=12, color=TEXT_DARK),
    showlegend=False,
), row=2, col=2)

# ── KPI banner annotations ─────────────────────────────────────
kpi_annotations = []
for i, (val, label, sub, color) in enumerate(kpis):
    kpi_annotations += [
        dict(x=kpi_x[i], y=1.13,  xref='paper', yref='paper',
             text=f'<b>{val}</b>', showarrow=False,
             font=dict(size=20, color=color), xanchor='center'),
        dict(x=kpi_x[i], y=1.085, xref='paper', yref='paper',
             text=label, showarrow=False,
             font=dict(size=11, color=TEXT_DARK), xanchor='center'),
        dict(x=kpi_x[i], y=1.052, xref='paper', yref='paper',
             text=sub, showarrow=False,
             font=dict(size=10, color=TEXT_MID), xanchor='center'),
    ]

# Subtitle annotation
kpi_annotations.append(
    dict(x=0.5, y=1.155, xref='paper', yref='paper',
         text='B2B SaaS AI feature impact  ·  835,000+ records  ·  42,777 users',
         showarrow=False,
         font=dict(size=10, color=TEXT_MID), xanchor='center')
)

# ── Layout ─────────────────────────────────────────────────────
axis_style = dict(
    gridcolor=GRID,
    linecolor=GRID,
    zerolinecolor=GRID,
    tickfont=dict(size=10, color=TEXT_MID),
)

fig.update_layout(
    title=dict(
        text='SmartAssist — Executive Dashboard',
        font=dict(size=18, color=TEXT_DARK, family='Helvetica, Arial, sans-serif'),
        x=0.5, y=0.98,
    ),
    plot_bgcolor=WHITE,
    paper_bgcolor=BG,
    font=dict(family='Helvetica, Arial, sans-serif', size=11, color=TEXT_DARK),
    height=820, width=1100,
    margin=dict(t=138, b=50, l=60, r=60),
    annotations=kpi_annotations,
)

fig.update_xaxes(**axis_style)
fig.update_yaxes(**axis_style)
fig.update_yaxes(ticksuffix='%', range=[0, 28], row=1, col=1)
fig.update_yaxes(ticksuffix='%', range=[0, 22], row=2, col=1)
fig.update_yaxes(ticksuffix='h', range=[0, avg_res_ai * 1.3], row=2, col=2)

# ── Export & show ──────────────────────────────────────────────
fig.write_image('../outputs/executive_dashboard.png', scale=2)
print('Dashboard saved to ../outputs/executive_dashboard.png')
fig.show()

Dashboard saved to ../outputs/executive_dashboard.png


---
## 9. Executive Summary and Recommendations

**To:** Alex Chen, VP of Product

**From:** Data Team

**Re:** SmartAssist Board Meeting Analysis

---

### Key Findings

| # | Finding | Data Point |
|---|---------|------------|
| 1 | Overall adoption is low | ~12.8% of users have tried SmartAssist |
| 2 | Enterprise adoption is critically low | Only ~4.7% vs ~20% for Starter tier |
| 3 | Severe stickiness problem | ~76% of users try it once and never return |
| 4 | Heavy usage correlates with higher churn | 16.4% churn vs 8.7% overall baseline |
| 5 | SmartAssist drives support costs up | AI tickets take ~2.7x longer to resolve |

---

### Recommendations

1. **Do not double down on scaling yet.** The current product experience needs fixing first.
2. **Fix the first-run experience.** 76% abandonment after one use points to an onboarding or value-delivery failure.
3. **Investigate why heavy users churn.** Is the feature buggy? Slow? Misleading? Conduct qualitative interviews.
4. **Prioritize Enterprise adoption.** This is our highest-value segment with the lowest engagement -- understand their blockers.
5. **Reduce support burden.** AI tickets take significantly longer to resolve -- invest in better self-serve documentation and in-product guidance.